# ID3 Decision Tree

ID3 (Iterative Dichotomiser 3) is one of the earliest and most influential decision tree algorithms. It uses information gain (based on entropy) to recursively split data and build a decision tree.

## Key Concepts:

- **Information Gain**: Uses entropy and information gain for attribute selection
- **Greedy Approach**: Selects the best attribute at each step (greedy algorithm)
- **Categorical Only**: Originally designed for categorical attributes only
- **Top-Down**: Builds tree in a top-down, recursive manner
- **No Pruning**: Original ID3 does not include pruning

## When to Use:

- When working with purely categorical data
- When you want a simple, interpretable model
- When you need to understand the decision process
- For educational purposes to understand decision tree fundamentals

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder

# Set style for better visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

## ID3 Algorithm Implementation

First, let's implement the ID3 algorithm from scratch to understand how it works.

In [ ]:
class ID3:
    """ID3 Decision Tree Algorithm Implementation"""
    
    def __init__(self, max_depth=None):
        self.max_depth = max_depth
        self.tree = None
    
    def entropy(self, y):
        """Calculate entropy of a set of labels"""
        unique_labels, counts = np.unique(y, return_counts=True)
        probabilities = counts / len(y)
        entropy = -np.sum(probabilities * np.log2(probabilities + 1e-10))
        return entropy
    
    def information_gain(self, X, y, feature_idx):
        """Calculate information gain for a given feature"""
        # Total entropy
        total_entropy = self.entropy(y)
        
        # Calculate weighted entropy after split
        values, counts = np.unique(X[:, feature_idx], return_counts=True)
        weighted_entropy = 0
        
        for value, count in zip(values, counts):
            mask = X[:, feature_idx] == value
            subset_y = y[mask]
            weighted_entropy += (count / len(y)) * self.entropy(subset_y)
        
        return total_entropy - weighted_entropy
    
    def best_feature(self, X, y, features):
        """Find the best feature to split on"""
        best_gain = -1
        best_feature = None
        
        for feature_idx in features:
            gain = self.information_gain(X, y, feature_idx)
            if gain > best_gain:
                best_gain = gain
                best_feature = feature_idx
        
        return best_feature, best_gain
    
    def build_tree(self, X, y, features, depth=0):
        """Recursively build the decision tree"""
        # Check stopping conditions
        if len(np.unique(y)) == 1:
            return {'class': y[0]}
        
        if len(features) == 0 or (self.max_depth and depth >= self.max_depth):
            return {'class': np.bincount(y).argmax()}
        
        # Find best feature
        best_feature_idx, best_gain = self.best_feature(X, y, features)
        
        if best_gain == 0:
            return {'class': np.bincount(y).argmax()}
        
        # Create node
        tree = {'feature': best_feature_idx, 'gain': best_gain, 'branches': {}}
        
        # Create branches
        remaining_features = [f for f in features if f != best_feature_idx]
        values = np.unique(X[:, best_feature_idx])
        
        for value in values:
            mask = X[:, best_feature_idx] == value
            X_subset = X[mask]
            y_subset = y[mask]
            
            if len(y_subset) == 0:
                tree['branches'][value] = {'class': np.bincount(y).argmax()}
            else:
                tree['branches'][value] = self.build_tree(
                    X_subset, y_subset, remaining_features, depth + 1
                )
        
        return tree
    
    def fit(self, X, y, feature_names=None):
        """Train the ID3 tree"""
        self.feature_names = feature_names or [f"Feature_{i}" for i in range(X.shape[1])]
        features = list(range(X.shape[1]))
        self.tree = self.build_tree(X, y, features)
    
    def predict_single(self, x, tree):
        """Predict a single sample"""
        if 'class' in tree:
            return tree['class']
        
        feature_idx = tree['feature']
        value = x[feature_idx]
        
        if value in tree['branches']:
            return self.predict_single(x, tree['branches'][value])
        else:
            # Handle unseen values
            # Return the most common class from all branches
            classes = [branch['class'] if 'class' in branch else 0 
                      for branch in tree['branches'].values()]
            return max(set(classes), key=classes.count)
    
    def predict(self, X):
        """Predict multiple samples"""
        return np.array([self.predict_single(x, self.tree) for x in X])
    
    def print_tree(self, tree=None, indent="", feature_names=None):
        """Print the tree structure"""
        if tree is None:
            tree = self.tree
            feature_names = self.feature_names
        
        if 'class' in tree:
            print(f"{indent}Class: {tree['class']}")
            return
        
        feature_name = feature_names[tree['feature']]
        print(f"{indent}{feature_name} (IG: {tree['gain']:.4f})")
        
        for value, branch in tree['branches'].items():
            print(f"{indent}  ├── {value}:")
            self.print_tree(branch, indent + "  │  ", feature_names)

## Dataset 1: Play Tennis Dataset

Classic example dataset for ID3 algorithm.

In [ ]:
# Create Play Tennis dataset
data = {
    'Outlook': ['Sunny', 'Sunny', 'Overcast', 'Rain', 'Rain', 'Rain', 'Overcast', 'Sunny', 'Sunny', 'Rain', 'Sunny', 'Overcast', 'Overcast', 'Rain'],
    'Temperature': ['Hot', 'Hot', 'Hot', 'Mild', 'Cool', 'Cool', 'Cool', 'Mild', 'Cool', 'Mild', 'Mild', 'Mild', 'Hot', 'Mild'],
    'Humidity': ['High', 'High', 'High', 'High', 'Normal', 'Normal', 'Normal', 'High', 'Normal', 'Normal', 'Normal', 'High', 'Normal', 'High'],
    'Wind': ['Weak', 'Strong', 'Weak', 'Weak', 'Weak', 'Strong', 'Strong', 'Weak', 'Weak', 'Weak', 'Strong', 'Strong', 'Weak', 'Strong'],
    'PlayTennis': ['No', 'No', 'Yes', 'Yes', 'Yes', 'No', 'Yes', 'No', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'No']
}

df = pd.DataFrame(data)
print("Play Tennis Dataset:")
print(df)
print(f"\nDataset shape: {df.shape}")

In [ ]:
# Encode categorical variables
le_dict = {}
df_encoded = df.copy()

for column in df.columns:
    le = LabelEncoder()
    df_encoded[column] = le.fit_transform(df[column])
    le_dict[column] = le

print("Encoded Dataset:")
print(df_encoded)

# Show encoding mappings
print("\nEncoding Mappings:")
for column, le in le_dict.items():
    print(f"{column}: {dict(zip(le.classes_, le.transform(le.classes_)))}")

In [ ]:
# Prepare data for ID3
X = df_encoded.drop('PlayTennis', axis=1).values
y = df_encoded['PlayTennis'].values
feature_names = df.drop('PlayTennis', axis=1).columns.tolist()

# Train ID3
id3 = ID3(max_depth=5)
id3.fit(X, y, feature_names)

# Print the tree
print("ID3 Decision Tree Structure:")
print("=" * 50)
id3.print_tree()

In [ ]:
# Make predictions
y_pred = id3.predict(X)
accuracy = accuracy_score(y, y_pred)

print(f"Training Accuracy: {accuracy:.4f}")
print(f"\nActual vs Predicted:")
for i in range(len(y)):
    actual = le_dict['PlayTennis'].inverse_transform([y[i]])[0]
    predicted = le_dict['PlayTennis'].inverse_transform([y_pred[i]])[0]
    print(f"Sample {i+1}: Actual={actual}, Predicted={predicted}")

## Using scikit-learn's DecisionTreeClassifier

scikit-learn's DecisionTreeClassifier with 'entropy' criterion implements ID3-like behavior.

In [ ]:
# Train sklearn's Decision Tree with entropy (ID3-like)
dt_id3 = DecisionTreeClassifier(
    criterion='entropy',
    max_depth=5,
    random_state=42
)

dt_id3.fit(X, y)
y_pred_sklearn = dt_id3.predict(X)
accuracy_sklearn = accuracy_score(y, y_pred_sklearn)

print(f"Sklearn Decision Tree Accuracy: {accuracy_sklearn:.4f}")

In [ ]:
# Visualize sklearn tree
plt.figure(figsize=(15, 10))
plot_tree(dt_id3, 
          feature_names=feature_names,
          class_names=['No', 'Yes'],
          filled=True,
          rounded=True,
          fontsize=10)
plt.title('ID3-like Decision Tree (sklearn) - Play Tennis', fontsize=16)
plt.tight_layout()
plt.show()

## Dataset 2: Car Evaluation Dataset

In [ ]:
# Create a simplified car evaluation dataset
car_data = {
    'Buying': ['vhigh', 'vhigh', 'vhigh', 'vhigh', 'vhigh', 'med', 'med', 'med', 'low', 'low', 'low', 'low'],
    'Maint': ['vhigh', 'vhigh', 'med', 'low', 'low', 'vhigh', 'vhigh', 'med', 'vhigh', 'vhigh', 'med', 'low'],
    'Doors': ['2', '2', '2', '2', '3', '2', '2', '2', '2', '3', '3', '3'],
    'Persons': ['2', '2', '2', '4', '4', '2', '4', '4', '4', '4', '4', '4'],
    'Lug_boot': ['small', 'med', 'big', 'small', 'med', 'small', 'med', 'big', 'small', 'med', 'big', 'big'],
    'Safety': ['low', 'med', 'high', 'low', 'med', 'low', 'med', 'high', 'low', 'med', 'high', 'high'],
    'Class': ['unacc', 'unacc', 'acc', 'unacc', 'acc', 'unacc', 'acc', 'good', 'unacc', 'acc', 'good', 'vgood']
}

car_df = pd.DataFrame(car_data)
print("Car Evaluation Dataset:")
print(car_df)
print(f"\nDataset shape: {car_df.shape}")

In [ ]:
# Encode car dataset
car_le_dict = {}
car_df_encoded = car_df.copy()

for column in car_df.columns:
    le = LabelEncoder()
    car_df_encoded[column] = le.fit_transform(car_df[column])
    car_le_dict[column] = le

# Prepare data
X_car = car_df_encoded.drop('Class', axis=1).values
y_car = car_df_encoded['Class'].values
car_feature_names = car_df.drop('Class', axis=1).columns.tolist()

# Split data
X_train_car, X_test_car, y_train_car, y_test_car = train_test_split(
    X_car, y_car, test_size=0.3, random_state=42, stratify=y_car
)

print(f"Training samples: {X_train_car.shape[0]}")
print(f"Test samples: {X_test_car.shape[0]}")

In [ ]:
# Train ID3 on car dataset
id3_car = ID3(max_depth=5)
id3_car.fit(X_train_car, y_train_car, car_feature_names)

# Make predictions
y_pred_car = id3_car.predict(X_test_car)
accuracy_car = accuracy_score(y_test_car, y_pred_car)

print(f"Test Accuracy: {accuracy_car:.4f}")

# Print tree
print("\nID3 Tree for Car Evaluation:")
print("=" * 50)
id3_car.print_tree()

## Information Gain Calculation

Let's manually calculate information gain for the Play Tennis dataset.

In [ ]:
# Calculate information gain for each feature
print("Information Gain for each feature:")
print("=" * 40)

for i, feature in enumerate(feature_names):
    gain = id3.information_gain(X, y, i)
    print(f"{feature}: {gain:.4f}")

print(f"\nBest feature: {feature_names[id3.best_feature(X, y, list(range(X.shape[1])))[0]]}")

## Cross-Validation

In [ ]:
# Use sklearn for cross-validation
dt_cv = DecisionTreeClassifier(criterion='entropy', max_depth=5, random_state=42)
cv_scores = cross_val_score(dt_cv, X_car, y_car, cv=3)

print(f"Cross-validation scores: {cv_scores}")
print(f"Mean CV accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")

## Predict on New Data

In [ ]:
# Function to predict play tennis
def predict_play_tennis(outlook, temperature, humidity, wind):
    # Encode the input
    outlook_enc = le_dict['Outlook'].transform([outlook])[0]
    temp_enc = le_dict['Temperature'].transform([temperature])[0]
    humidity_enc = le_dict['Humidity'].transform([humidity])[0]
    wind_enc = le_dict['Wind'].transform([wind])[0]
    
    sample = np.array([[outlook_enc, temp_enc, humidity_enc, wind_enc]])
    prediction = id3.predict(sample)[0]
    
    return le_dict['PlayTennis'].inverse_transform([prediction])[0]

# Test predictions
test_cases = [
    ('Sunny', 'Hot', 'High', 'Weak'),
    ('Overcast', 'Hot', 'High', 'Weak'),
    ('Rain', 'Mild', 'Normal', 'Weak')
]

for outlook, temp, humidity, wind in test_cases:
    result = predict_play_tennis(outlook, temp, humidity, wind)
    print(f"{outlook}, {temp}, {humidity}, {wind} -> Play Tennis: {result}")

## Summary

### Key Takeaways:

1. **Information Gain**: Uses entropy-based information gain for attribute selection
2. **Greedy Algorithm**: Makes locally optimal choices at each step
3. **Categorical Data**: Originally designed for categorical attributes only
4. **Top-Down Induction**: Builds tree recursively in a top-down manner
5. **No Pruning**: Original ID3 doesn't include pruning (can overfit)

### Advantages:
- Simple and intuitive algorithm
- Highly interpretable
- Works well with categorical data
- Fast to train on small datasets
- No need for feature scaling

### Limitations:
- Only handles categorical attributes (original version)
- Can overfit without pruning
- Greedy approach may not find globally optimal tree
- Sensitive to small changes in data
- Doesn't handle missing values well (original version)
- May create deep trees with many branches

### ID3 vs C4.5:

- **ID3**: Categorical only, no pruning, information gain
- **C4.5**: Handles continuous attributes, includes pruning, uses gain ratio